In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 15, The intrinsic engine

Companion notebook to [15_intrinsic_engine.md](15_intrinsic_engine.md). `intrinsic_engine()` bundles the textbook expansions of `ι_X`, `L_X`, `d` on a `MultiEval` plus the multi-eval book-keeping helpers; `prove_intrinsic_equivalence` drives them to fix-point.

## The base bundle

Seven rules: three intrinsic operator expansions plus four `MultiEval` helpers. Match order matters, operator-specific rules fire before head-linearity scans the wrapper.

In [ ]:
from jacopy.calculus.intrinsic_engine import intrinsic_engine

eng = intrinsic_engine()
print(f"rules : {len(eng.definitions)}")
for r in eng.definitions:
    print(f"  - {r.name}")

## `prove_intrinsic_equivalence` on `ι² = 0`

Vector fields are constructed via `Derivation(name, 0)` (degree 0), the intrinsic rules look for that shape rather than a generic `Symbol` with a `Graded` declaration.

In [ ]:
from jacopy.algebra.derivation import Act, Derivation
from jacopy.calculus.interior import interior
from jacopy.calculus.intrinsic_engine import prove_intrinsic_equivalence
from jacopy.core.expr import Symbol, Integer
from jacopy.core.multi_eval import multi_eval

omega = Symbol("ω")
X, Y = Derivation("X", 0), Derivation("Y", 0)

lhs = multi_eval(Act(interior(X), Act(interior(X), omega)), Y)
chain = prove_intrinsic_equivalence(lhs, Integer(0))
print(f"ι_X ι_X ω = 0 closes in {len(chain)} steps")
for s in chain.steps:
    print(f"  - {s.rule}")

## Cartan magic on a 2-form

The flagship 12.A test: `(ι_X d + d ι_X) ω = L_X ω` on a 2-form, evaluated on `(Y, Z)`. The base bundle alone closes this in twelve named steps.

In [ ]:
from jacopy.calculus.exterior_d import d as default_d
from jacopy.calculus.lie_derivative import lie_derivative
from jacopy.core.expr import Sum

X, Y, Z = (Derivation(s, 0) for s in ("X", "Y", "Z"))

lhs = Sum(
    multi_eval(Act(interior(X), Act(default_d, omega)), Y, Z),
    multi_eval(Act(default_d, Act(interior(X), omega)), Y, Z),
)
rhs = multi_eval(Act(lie_derivative(X), omega), Y, Z)

chain = prove_intrinsic_equivalence(lhs, rhs)
print(f"Cartan magic closes in {len(chain)} steps")

## Closure-complete bundle

`intrinsic_engine_with_closure()` adds four 12.A.6 rules that fold post-expansion residues, VF-commutator, bracket antisymmetry / Jacobi, the iota-as-scalar 1-form bridge. Together they close `[L_X, ι_Y] ω = ι_{[X,Y]_VF} ω` and `d² = 0` on 1- / 2-forms.

In [ ]:
from jacopy.algebra.lie_bracket_vf import lie_bracket_vf
from jacopy.calculus.intrinsic_engine import intrinsic_engine_with_closure
from jacopy.core.expr import Neg

XY = lie_bracket_vf(X, Y)

lhs = Sum(
    multi_eval(Act(lie_derivative(X), Act(interior(Y), omega)), Z),
    Neg(multi_eval(Act(interior(Y), Act(lie_derivative(X), omega)), Z)),
)
rhs = multi_eval(Act(interior(XY), omega), Z)

chain = prove_intrinsic_equivalence(
    lhs, rhs, engine=intrinsic_engine_with_closure(),
)
print(f"[L_X, ι_Y] ω closes in {len(chain)} steps")

## `IntrinsicFormulaRecognizer`, shape inspection

Pure-shape inspection of `MultiEval(Act(op, ω), Y_1, …)`: given an expression, returns the operator label, the form, the vector field (if any), and the eval slots, without running any rewriting. Use it when you want to *dispatch* on shape without committing to a closure.

In [ ]:
from jacopy.calculus.intrinsic_engine import IntrinsicFormulaRecognizer

expr = multi_eval(Act(lie_derivative(X), omega), Y, Z)
match = IntrinsicFormulaRecognizer().recognize(expr)
print(f"operator     : {match.operator}")
print(f"vector_field : {match.vector_field}")
print(f"omega        : {match.omega}")
print(f"args         : {match.args}")

## When the intrinsic engine is the wrong choice

**Use it for:** equalities involving `MultiEval(Act(op, ω), Y_1, …)` shapes, when you want the textbook intrinsic-formula transcript.

**Don't use it for:** generic operator-equation work (raw `L²`, `d²` without multi-eval), that belongs to `prove_equivalence` with `default_engine`. Problem-specific axioms belong in the closure-axiom layer (tutorial 13) or a problem wrapper (tutorial 14).

**Known failure mode:** `d²` and `[L_X, L_Y]` on a 3-form or higher don't close, the 12.A.6 closure axioms are calibrated for 1- / 2-forms. Tutorial 10 walks the diagnostic surface for those residues.

## Summary

* `intrinsic_engine()`, 7 base rules: 3 intrinsic operator expansions + 4 multi-eval helpers.
* `intrinsic_engine_with_closure()`, adds 4 closure rules to fold post-expansion residues; closes `[L_X, ι_Y] ω`, `d² = 0`, `[L_X, L_Y] ω` on 1- / 2-forms.
* `prove_intrinsic_equivalence`, runs the engine to fix-point, returns a `ProofChain`.
* `IntrinsicFormulaRecognizer`, pure-shape inspector for intrinsic-operator multi-evals, no rewriting.